In [1]:
import pandas as pd
import numpy as np
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
import spacy
import os

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn import metrics

from sklearn.metrics import classification_report

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

from string import punctuation
from collections import Counter

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
RANDOM_SEED = 12345

In [3]:
# Список файлов
file_list = [
    "00226 Дина.csv",
    "sarcasm00026.xlsx",
    "Сарказм-00126-Полина.xlsx",

]

# Путь к папке с файлами
folder_path = r"/content/datas"

# Создаем пустой список для хранения датафреймов
dfs = []

# Загружаем каждый файл
for file in file_list:
    file_path = os.path.join(folder_path, file)

    if file.endswith('.csv'):
        df = pd.read_csv(file_path, encoding = 'cp1251', delimiter=';')
    elif file.endswith('.xlsx'):
        df = pd.read_excel(file_path)
    else:
        continue  # пропускаем неизвестные форматы

    dfs.append(df)

# Объединяем все датафреймы
combined_df = pd.concat(dfs, ignore_index=True)
combined_df = combined_df.dropna(subset=['sarcasm']).reset_index(drop=True)
combined_df['sarcasm'] = combined_df['sarcasm'].astype(int)

# Так как эти датасеты имеют свою кодировку загрузим их отдельно
df_kate = pd.read_csv(r'/content/datas/пипипу.csv')
df_alex = pd.read_csv(r'/content/datas/dataset_all_data.csv')
df_lisa = pd.read_csv(r'/content/datas/flatten_res_00426.csv')

dfs = [combined_df, df_alex, df_kate, df_lisa]

# Задаём полный набор колонок, который хотим в финале
all_cols = ['text', 'genre', 'gender', 'age', 'exp', 'sarcasm']

cleaned = []
for df in dfs:
    df = df.copy()

    # Преобразуем 0.0/1.0 в 0/1
    df['sarcasm'] = df['sarcasm'].astype(int)

    # Если какие‑то колонки отсутствуют — создаём их с NaN
    for col in all_cols:
        if col not in df.columns:
            df[col] = np.nan

    # Оставляем только нужный порядок колонок
    df = df[all_cols]
    cleaned.append(df)

# Склеиваем всё вместе
final_df = pd.concat(cleaned, ignore_index=True)
final_df = final_df[final_df['sarcasm'].isin([0, 1])].reset_index(drop=True)
data = final_df

print(data.shape)
print(data['sarcasm'].value_counts())

(34185, 6)
sarcasm
0    32551
1     1634
Name: count, dtype: int64


In [4]:
# Общая информация и пропуски
print("Общая информация по DataFrame:")
print(data.info())
print("\nКоличество пропусков по столбцам:")
print(data.isna().sum())

# Распределение сарказма
print("\nРаспределение меток sarcasm:")
print(data['sarcasm'].value_counts(dropna=False))
print("\nДоля каждой метки sarcasm:")
print(data['sarcasm'].value_counts(normalize=True))

# Анализ по другим полям
for col in ['genre', 'gender', 'exp']:
    print(f"\nРаспределение по {col}:")
    print(data[col].value_counts(dropna=False))
    print(f"\nДоля по {col}:")
    print(data[col].value_counts(normalize=True, dropna=False))

# Статистика по длине текста
data['text_length_chars'] = data['text'].str.len()
data['text_length_words'] = data['text'].str.split().str.len()

print("\nСтатистика длины текста (символы):")
print(data['text_length_chars'].describe())
print("\nСтатистика длины текста (слова):")
print(data['text_length_words'].describe())

# Сравнение длины текста в саркастичных/несаркастичных примерах
print("\nДлина текста по классам sarcasm (символы):")
print(data.groupby('sarcasm')['text_length_chars'].describe())

print("\nДлина текста по классам sarcasm (слова):")
print(data.groupby('sarcasm')['text_length_words'].describe())

Общая информация по DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34185 entries, 0 to 34184
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   text     34185 non-null  object 
 1   genre    25041 non-null  object 
 2   gender   9144 non-null   float64
 3   age      9144 non-null   float64
 4   exp      67 non-null     object 
 5   sarcasm  34185 non-null  int64  
dtypes: float64(2), int64(1), object(3)
memory usage: 1.6+ MB
None

Количество пропусков по столбцам:
text           0
genre       9144
gender     25041
age        25041
exp        34118
sarcasm        0
dtype: int64

Распределение меток sarcasm:
sarcasm
0    32551
1     1634
Name: count, dtype: int64

Доля каждой метки sarcasm:
sarcasm
0    0.952201
1    0.047799
Name: proportion, dtype: float64

Распределение по genre:
genre
social    25041
NaN        9144
Name: count, dtype: int64

Доля по genre:
genre
social    0.732514
NaN       0.267486
Name: pr

In [5]:
y = data['sarcasm']
X = data.drop('sarcasm', axis=1)

In [6]:
def feature_engineering(choice_transformer, choice_ngrams):
    # Обработка текстовых данных: либо TF-IDF, либо мешок слов
    text_features = 'text'
    if choice_transformer == 'tfidf':
        text_transformer = TfidfVectorizer(
            ngram_range=choice_ngrams,
            tokenizer=word_tokenize,
            stop_words='english'
        )
    else:
        text_transformer = CountVectorizer(
            ngram_range=choice_ngrams,
            tokenizer=word_tokenize,
            stop_words='english'
        )

    # Применяем трансформацию только к текстовому столбцу 'text'
    preprocessor = ColumnTransformer(
        transformers=[
            ("txt", text_transformer, text_features)
        ]
    )
    return preprocessor

In [7]:
def modelfit(model):
    model.fit(Xtrain, ytrain)

    # Предсказания меток
    ypredtest = model.predict(Xtest)
    ypredtrain = model.predict(Xtrain)

    # Предсказания вероятностей (для roc_auc_score)
    yprobtest = model.predict_proba(Xtest)
    yprobtrain = model.predict_proba(Xtrain)

    print('RESULTS:\nroc-auc_score:\n',
          #'train:', roc_auc_score(ytrain, yprobtrain, multi_class='ovr'),
          #'test:', roc_auc_score(ytest, yprobtest, multi_class='ovr'),
          '\nf1_score:\n',
          'train:', f1_score(ytrain, ypredtrain, average='macro'),
          'test:', f1_score(ytest, ypredtest, average='macro'),
          '\nclassification_report\ntrain:\n',
          classification_report(ytrain, ypredtrain),
          '\ntest:\n',
          classification_report(ytest, ypredtest)
         )

In [8]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, stratify=y, random_state = RANDOM_SEED)

ytrain = ytrain.squeeze()
ytest = ytest.squeeze()

preprocessor = feature_engineering('tfidf', (1, 1))

clfLR = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression(class_weight='balanced', random_state = RANDOM_SEED))]
)

clfSVC = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", SVC(class_weight='balanced', probability=True, random_state = RANDOM_SEED))]
)

## Логическая регрессия

In [9]:
modelfit(clfLR)

/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


RESULTS:
roc-auc_score:
 
f1_score:
 train: 0.7581502005251498 test: 0.6253853800546739 
classification_report
train:
               precision    recall  f1-score   support

           0       1.00      0.92      0.96     26041
           1       0.39      0.99      0.56      1307

    accuracy                           0.92     27348
   macro avg       0.69      0.96      0.76     27348
weighted avg       0.97      0.92      0.94     27348
 
test:
               precision    recall  f1-score   support

           0       0.98      0.87      0.92      6510
           1       0.21      0.69      0.33       327

    accuracy                           0.86      6837
   macro avg       0.60      0.78      0.63      6837
weighted avg       0.95      0.86      0.90      6837



## Базовый Берт

In [10]:
import torch
import torch.nn as nn
from transformers import BertModel, BertConfig

In [11]:
class CustomBertClassifier(nn.Module):
    def __init__(self,
                 pretrained_model_name: str = 'DeepPavlov/rubert-base-cased',
                 num_labels: int = 2,
                 hidden_dim: int = 768,
                 dropout_prob: float = 0.1):
        super().__init__()
        # Базовая модель берта без головы для маскированного языка
        self.bert = BertModel.from_pretrained(pretrained_model_name)

        # Кастомная голова
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_prob),
            nn.Linear(self.bert.config.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self,
                input_ids: torch.LongTensor,
                attention_mask: torch.Tensor = None,
                token_type_ids: torch.Tensor = None,
                labels: torch.LongTensor = None):
        # Получаем выходы из Берт
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        # Выбираем pooled_output
        pooled_output = outputs.pooler_output

        # Передаём через свою голову
        logits = self.classifier(pooled_output)

        # Если есть метки - считаем loss
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {
                'loss': loss,
                'logits': logits
            }
        return {'logits': logits}

In [14]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import BertTokenizer, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from dataclasses import dataclass

# Загружаем и обрабатываем
data['sarcasm'] = data['sarcasm'].astype(int)

train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    stratify=data['sarcasm'],
    random_state=42
)

# Токенизация
tokenizer = BertTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')

@dataclass
class NERFeatures:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    token_type_ids: torch.Tensor
    labels: torch.Tensor

class SarcasmDataset(TorchDataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['sarcasm'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        enc = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'token_type_ids': enc.get('token_type_ids', torch.zeros_like(enc['input_ids'])).squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Создаём датасеты
train_dataset = SarcasmDataset(train_df, tokenizer)
test_dataset  = SarcasmDataset(test_df, tokenizer)

# Загрузка модели
model = CustomBertClassifier(
    pretrained_model_name='DeepPavlov/rubert-base-cased',
    num_labels=2,
    hidden_dim=768,
    dropout_prob=0.1
)

# Метрики
def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro')
    }

# Параметры тренировки
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,

    # частота в шагах
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,

    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
)

# Трейнер
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Обучение и оценка
trainer.train()
results = trainer.evaluate()
print(results)
# Сохраняем лучшую модель
trainer.save_model('./best_model')

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Step,Training Loss
50,0.220700
100,0.160000
150,0.183200
200,0.166400
250,0.154800
300,0.140700
350,0.162500
400,0.135000
450,0.136800
500,0.156800


{'eval_loss': 0.17826814949512482, 'eval_accuracy': 0.9552435278630979, 'eval_f1': 0.7131567805673131, 'eval_runtime': 13.6872, 'eval_samples_per_second': 499.516, 'eval_steps_per_second': 7.817, 'epoch': 3.0}
